# Description

In this notebook, we benchmark different symbolic regression algorithms on the Korns benchmarks.

In [27]:
from __future__ import annotations
import time
from dataclasses import dataclass, asdict
from typing import Dict, Any, List, Optional, Protocol, Tuple
import numpy as np
import sympy as sp
import h5py

@dataclass(frozen=True)
class DatasetRecord:
    pid: str
    X: np.ndarray
    y: np.ndarray
    expr_gt: sp.Expr

def load_korns_hdf5(path: str) -> Dict[str, DatasetRecord]:
    out: Dict[str, DatasetRecord] = {}
    with h5py.File(path, "r") as f:
        for pid in f.keys():
            grp = f[pid]
            X = np.asarray(grp["X"][:], dtype=np.float64)
            y = np.asarray(grp["y"][:], dtype=np.float64).reshape(-1)

            # TODO: remove?
            if "expr_srepr" in grp.attrs:
                expr_gt = sp.sympify(grp.attrs["expr_srepr"])
            else:
                expr_gt = sp.sympify(grp.attrs["expr_str"])

            out[pid] = DatasetRecord(pid=pid, X=X, y=y, expr_gt=expr_gt)
    return out

def train_test_split(
    X: np.ndarray,
    y: np.ndarray,
    test_size: float = 0.2,
    seed: int = 0,
) -> Tuple[np.ndarray, np.ndarray, np.ndarray, np.ndarray]:
    assert X.ndim == 2 and y.ndim == 1 and X.shape[0] == y.shape[0]
    rng = np.random.default_rng(seed)
    n = X.shape[0]
    idx = np.arange(n)
    rng.shuffle(idx)

    n_test = int(round(test_size * n))
    test_idx = idx[:n_test]
    train_idx = idx[n_test:]

    return X[train_idx], X[test_idx], y[train_idx], y[test_idx]

@dataclass(frozen=True)
class Metrics:
    mse: float
    rmse: float
    mae: float
    r2: float

def compute_metrics(y_true: np.ndarray, y_pred: np.ndarray) -> Metrics:
    y_true = np.asarray(y_true, dtype=np.float64).reshape(-1)
    y_pred = np.asarray(y_pred, dtype=np.float64).reshape(-1)
    if y_true.shape != y_pred.shape:
        raise ValueError(f"Shape mismatch: y_true {y_true.shape} vs y_pred {y_pred.shape}")

    err = y_true - y_pred
    mse = float(np.mean(err**2))
    rmse = float(np.sqrt(mse))
    mae = float(np.mean(np.abs(err)))

    ss_res = float(np.sum(err**2))
    ss_tot = float(np.sum((y_true - float(np.mean(y_true)))**2))
    r2 = float(1.0 - ss_res / ss_tot) if ss_tot > 0 else float("nan")

    return Metrics(mse=mse, rmse=rmse, mae=mae, r2=r2)

@dataclass
class SRFitResult:
    expr: Optional[sp.Expr]
    y_pred_test: np.ndarray
    metadata: Dict[str, Any]

class SymbolicRegressor(Protocol):
    name:str

    def fit_predict(
        self,
        X_train: np.ndarray,
        y_train: np.ndarray,
        X_test: np.ndarray,
        feature_names: List[str],
    ) -> SRFitResult:
        ...

@dataclass
class DummyMeanRegressor:
    name: str = "dummy_mean"

    def fit_predict(
        self,
        X_train: np.ndarray,
        y_train: np.ndarray,
        X_test: np.ndarray,
        feature_names: List[str],
    ) -> SRFitResult:
        mu = float(np.mean(y_train))
        y_pred = np.full(shape=(X_test.shape[0],), fill_value=mu, dtype=np.float64)
        expr = sp.Float(mu)
        return SRFitResult(expr=expr, y_pred_test=y_pred, metadata={"mean": mu})

@dataclass
class RunConfig:
    hdf5_path: str = "korns_dataset.hdf5"
    test_size: float = 0.2
    split_seed: int = 0
    per_problem_seed_offset: int = 1000

@dataclass
class BenchmarkRow:
    pid: str
    algo: str
    runtime_sec: float
    mse: float
    rmse: float
    mae: float
    r2: float
    expr_str: str
    exact_match: Optional[bool] = None
    extra: Optional[Dict[str, Any]] = None

def run_benchmark(
    datasets: Dict[str, DatasetRecord],
    algorithms: List[SymbolicRegressor],
    config: RunConfig,
) -> List[BenchmarkRow]:
    feature_names = ["x0", "x1", "x2", "x3", "x4"]
    rows: List[BenchmarkRow] = []

    for pid, rec in sorted(datasets.items(), key=lambda kv: int(kv[0][1:])):
        split_seed = config.split_seed + config.per_problem_seed_offset + int(pid[1:])
        X_train, X_test, y_train, y_test = train_test_split(
            rec.X, rec.y, test_size=config.test_size, seed=split_seed
        )

        for algo in algorithms:
            t0 = time.time()
            fit_res = algo.fit_predict(X_train, y_train, X_test, feature_names)
            runtime = time.time() - t0

            m = compute_metrics(y_test, fit_res.y_pred_test)
            expr_str = str(fit_res.expr) if fit_res.expr is not None else ""

            rows.append(
                BenchmarkRow(
                    pid=pid,
                    algo=algo.name,
                    runtime_sec=float(runtime),
                    mse=m.mse,
                    rmse=m.rmse,
                    mae=m.mae,
                    r2=m.r2,
                    expr_str=expr_str,
                    exact_match=None,
                    extra=fit_res.metadata if fit_res.metadata else None,
                )
            )
    return rows

def save_results_csv(rows: List[BenchmarkRow], path: str) -> None:
    import csv
    fieldnames = list(asdict(rows[0]).keys()) if rows else []
    with open(path, "w", newline="") as f:
        w = csv.DictWriter(f, fieldnames=fieldnames)
        w.writeheader()
        for r in rows:
            d = asdict(r)

            if d.get("extra") is not None:
                d["extra"] = str(d["extra"])
            w.writerow(d)

cfg = RunConfig(hdf5_path="korns_dataset.hdf5", test_size=0.2, split_seed=0)
datasets = load_korns_hdf5(cfg.hdf5_path)
algos: List[SymbolicRegressor] = [
    DummyMeanRegressor(),
]

rows = run_benchmark(datasets, algos, cfg)
for r in rows[:5]:
    print(r)

save_results_csv(rows, "korns_benchmark_results_v0.csv")